# 📥 RAG Retrieval (Elasticsearch)

This notebook handles the **expensive** part of the RAG evaluation pipeline:  
connecting to Elasticsearch, running batch retrieval across strategies, and saving raw results to disk.

## Pipeline
1. **Load** the QA dataset (questions + ground-truth Wikipedia IDs)
2. **Connect** to the Elasticsearch index
3. **Retrieve** top-K documents for every question using each strategy
4. **Save** raw retrieval results as Parquet files

## Strategies
- **Dense Vector Search** (`approximation`): Semantic similarity via embeddings
- **BM25 Keyword Search** (`bm25`): Traditional full-text search
- **Hybrid Search** (`hybrid`): Combined vector + BM25

## Output
Results are saved to `{COLLECTION_ROOT}/{OUTPUT_NAME}/` as `results_{strategy}.parquet`.  
These files are consumed by `rag_evaluation.ipynb` for metric computation and visualization.

> **Note:** Run this notebook once (or when you change retrieval settings).  
> The evaluation notebook can be re-run cheaply on saved results.

## 1. Configuration

In [1]:

import os
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv

from rag.elasticsearch_rag_service import ElasticsearchRagService
from rag.utils import IndexingConfig
from config import DATA_DIR


load_dotenv()

# Elasticsearch Connection
ES_URL = "http://localhost:9200"
ES_USER = os.getenv("ELASTICSEARCH_USERNAME") or None
ES_PASSWORD = os.getenv("ELASTICSEARCH_PASSWORD") or None
OUTPUT_NAME = "all_qa_8k"
# Used for naming outputs, e.g., "wiki_full_all_qa_8k.parquet"

# Collection & Index
COLLECTION_NAME = "wiki_full_bil"
COLLECTION_ROOT = Path(DATA_DIR) / COLLECTION_NAME
QUESTIONS_PATH = COLLECTION_ROOT / f"{OUTPUT_NAME}.parquet"

# Embedding Configuration (MUST match indexing settings exactly!)
# EMBEDDING_MODEL = "infloat/multilingual-e5-large"  # Match rag_indexing.ipynb!
EMBEDDING_MODEL = "Lajavaness/bilingual-embedding-small"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 100
    
# Retrieval Configuration
STRATEGIES = ["bm25", "approximation"]  # Strategies to evaluate
TOP_K = 100  # Primary metric for reporting (e.g., Recall@10 for popularity decile analysis)
K_VALUES_DETAILED = [1,3,5,10]  # K values for investigation/curves (e.g., 1-20 to see full trend)
MAX_QUESTIONS = None  # Set to limit for testing (e.g., 100), None for all

# kNN Search Configuration
# num_candidates controls how many HNSW nodes are explored per kNN query.
# Lower = faster but lower recall. 100 is a good balance for large remote indices.
NUM_CANDIDATES = 1000  # Reduced from 1000 for remote server performance

# Batching: number of kNN queries bundled into a single _msearch HTTP request.
# Higher = fewer round-trips to the remote server. 200 is a good default.
MSEARCH_BATCH_SIZE = 5  # Batch multiple kNN queries per HTTP request

EMBED_BATCH_SIZE = 512   # Smaller batches reduce peak CPU memory during embedding
SEARCH_WORKERS = 5       # Parallel _msearch batches (not individual queries)


# Output Configuration
OUTPUT_FOLDER = "wiki_full_bil_100"  # Folder name for storing all results
RESULTS_DIR = COLLECTION_ROOT / OUTPUT_FOLDER
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Config loaded")
print(f"  Index: {COLLECTION_NAME} @ {ES_URL}")
print(f"  Questions: {QUESTIONS_PATH}")
print(f"  Embedding Model: {EMBEDDING_MODEL}")
print(f"  Top-K (Primary Metric): {TOP_K}")
print(f"  K-Values for Investigation: {list(K_VALUES_DETAILED)}")
print(f"  kNN num_candidates: {NUM_CANDIDATES}")
print(f"  msearch batch size: {MSEARCH_BATCH_SIZE}")
print(f"  Will retrieve: {max(TOP_K, max(K_VALUES_DETAILED))} documents per query")
print(f"  Results: {RESULTS_DIR}")


✓ Config loaded
  Index: wiki_full_bil @ http://localhost:9200
  Questions: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/all_qa_8k.parquet
  Embedding Model: Lajavaness/bilingual-embedding-small
  Top-K (Primary Metric): 100
  K-Values for Investigation: [1, 3, 5, 10]
  kNN num_candidates: 1000
  msearch batch size: 5
  Will retrieve: 100 documents per query
  Results: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100


## 2. Load Questions

Load the QA dataset containing questions, ground-truth Wikipedia IDs, popularity scores, and decile labels.

In [2]:
print("Loading questions...")
qa_df = pd.read_parquet(QUESTIONS_PATH)

qa_df = qa_df.dropna(subset=["question_text"])

# Normalize Wikipedia IDs to string format for matching
qa_df["wikipedia_id"] = qa_df["wikipedia_id"].astype(str).str.strip()

# Limit for testing if specified
if MAX_QUESTIONS:
    qa_df = qa_df.sample(n=min(MAX_QUESTIONS, len(qa_df)), random_state=42)
    print(f"  Limited to {len(qa_df)} questions for testing")

print(f"✓ Loaded {len(qa_df):,} questions")
print(f"  Unique docs: {qa_df['wikipedia_id'].nunique():,}")
print(f"  Datasets: {qa_df['dataset'].value_counts().to_dict() if 'dataset' in qa_df.columns else 'N/A'}")

# Display sample
print("\nSample questions:")
display(qa_df.head(3))

Loading questions...
✓ Loaded 50,575 questions
  Unique docs: 44,857
  Datasets: {'trex': 13080, 'pop_qa': 11125, 'hotpot_qa': 9842, 'natural_questions': 6432, 'trivia_qa': 5465, 'fever': 4631}

Sample questions:


,question_id,question_text,answer_texts,wikipedia_id,wikipedia_title,popularity_avg,popularity_rank,dataset,pop_decile_unweighted,pop_decile_chunk_weighted,decile
0,3775083,Who was the producer of Competition?,[American Film Manufacturing Company],14902716,Competition (1915 film),18.1250,4.461622e+06,pop_qa,2,0,0
1,5920612,Who was the director of The Day?,[Alfred Rolfe],32987749,The Day (1914 film),19.5625,4.337318e+06,pop_qa,2,0,0
2,488360,What is Carlos María Ramírez's occupation?,[journalist],37748489,Carlos María Ramírez,19.6250,4.353555e+06,pop_qa,2,0,0


## 3. Connect to Elasticsearch Index

Initialize the `ElasticsearchRagService` and connect to the pre-built index.  
The embedding model and chunking parameters **must** match what was used during indexing (`rag_indexing.ipynb`).

In [3]:
print("Connecting to Elasticsearch index...")

config = IndexingConfig(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    embedding_provider="modal",
    embedding_model=EMBEDDING_MODEL,
    gpu_batch_size=512,
    request_batch_size=1024,
    normalise_embeddings=True,
    trust_remote_code=True,
    use_progress=False
)

# Initialize service (strategy doesn't matter for loading, we'll override at query time)
service = ElasticsearchRagService(
    config=config,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
    strategy="hybrid"
)

service.load_index(COLLECTION_NAME)


print(f"✓ Connected to Elasticsearch index: {COLLECTION_NAME}")

Connecting to Elasticsearch index...
✓ Connected to Elasticsearch index: wiki_full_bil


## 4. Run Batch Retrieval

For each strategy, retrieve the top-K documents for every question.  
This is the most **time-consuming** step — results are cached to Parquet afterwards.

Each result row contains:
- `topk_ids`: Wikipedia IDs of retrieved documents
- `topk_scores`: Relevance scores
- `topk_popularities`: Popularity values of retrieved documents

In [7]:

print("Running retrieval for all strategies...\n")

from helpers.decile_utils import COL_DECILE_UNWEIGHTED, COL_DECILE_CHUNK_WEIGHTED

results_by_strategy = {}

for strategy in STRATEGIES:
    print(f"{'='*60}")
    print(f"Strategy: {strategy.upper()}")
    print(f"{'='*60}")

    # Skip if results already saved from a previous run
    output_path = RESULTS_DIR / f"results_{strategy}.parquet"
    if output_path.exists():
        print(f"⏭  Skipping {strategy} — results already exist at {output_path}\n")
        results_by_strategy[strategy] = pd.read_parquet(output_path)
        continue

    # Only use num_candidates / msearch for vector/hybrid strategies
    nc = NUM_CANDIDATES if strategy in ("approximation", "hybrid") else None
    ms = MSEARCH_BATCH_SIZE if strategy in ("approximation", "hybrid") else 0

    all_results = service.batch_retrieve(
        questions=qa_df["question_text"].tolist(),
        top_k=max(TOP_K, max(K_VALUES_DETAILED)),
        strategy=strategy,
        num_candidates=nc,
        embed_batch_size=EMBED_BATCH_SIZE,
        search_workers=SEARCH_WORKERS,
        msearch_batch_size=ms,
    )
    
    # Process results
    rows = []
    for idx, (question_row, retrieved_docs) in enumerate(zip(qa_df.itertuples(), all_results)):
        expected_id = str(question_row.wikipedia_id).strip()
        
        retrieved_ids = []
        retrieved_scores = []
        retrieved_popularities = []
        
        for doc, score in retrieved_docs:
            raw_id = doc.metadata.get("wikipedia_id", doc.metadata.get("id", ""))
            doc_id = str(int(float(raw_id))) if raw_id not in (None, "") else ""
            retrieved_ids.append(doc_id)
            retrieved_scores.append(score)
            retrieved_popularities.append(doc.metadata.get("popularity_avg"))
        
        rows.append({
            "question": question_row.question_text,
            "wikipedia_id": expected_id,
            "wikipedia_title": getattr(question_row, "wikipedia_title", None),
            "popularity_avg": getattr(question_row, "popularity_avg", None),
            "dataset": getattr(question_row, "dataset", None),
            COL_DECILE_UNWEIGHTED: getattr(question_row, COL_DECILE_UNWEIGHTED, -1),
            COL_DECILE_CHUNK_WEIGHTED: getattr(question_row, COL_DECILE_CHUNK_WEIGHTED, -1),
            "decile": getattr(question_row, "decile", -1),
            "topk_ids": retrieved_ids,
            "topk_scores": retrieved_scores,
            "topk_popularities": retrieved_popularities,
        })
    
    results_df = pd.DataFrame(rows)
    results_by_strategy[strategy] = results_df

    # Save immediately after this strategy finishes
    results_df.to_parquet(output_path)
    print(f"  💾 Saved {strategy} → {output_path} ({len(results_df):,} rows)\n")

ALL_STRATEGIES = list(results_by_strategy.keys())
print(f"✅ All retrieval complete! Tested {len(ALL_STRATEGIES)} configurations:")
print(f"   {', '.join(ALL_STRATEGIES)}")


Running retrieval for all strategies...

Strategy: BM25
⏭  Skipping bm25 — results already exist at /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100/results_bm25.parquet

Strategy: APPROXIMATION


Retrieving (approximation): 100%|██████████| 50575/50575 [00:27<00:00, 1807.27q/s]


  💾 Saved approximation → /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100/results_approximation.parquet (50,575 rows)

✅ All retrieval complete! Tested 2 configurations:
   bm25, approximation


## 5. Save Raw Results

Save retrieval results as Parquet files — one per strategy.  
These are the input for `rag_evaluation.ipynb`.

In [5]:
print("Saving retrieval results...\n")

for strategy in ALL_STRATEGIES:
    results_df = results_by_strategy[strategy]
    output_path = RESULTS_DIR / f"results_{strategy}.parquet"
    results_df.to_parquet(output_path)
    print(f"  ✓ Saved {strategy}: {output_path} ({len(results_df):,} rows)")

print(f"\n✅ All results saved to: {RESULTS_DIR}")
print(f"\nNext step: Open rag_evaluation.ipynb to compute metrics and generate visualizations.")

Saving retrieval results...

  ✓ Saved bm25: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100/results_bm25.parquet (50,575 rows)
  ✓ Saved approximation: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100/results_approximation.parquet (50,575 rows)

✅ All results saved to: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100

Next step: Open rag_evaluation.ipynb to compute metrics and generate visualizations.


## 6. Generate Metadata

Calculate decile boundaries (both unweighted and chunk-weighted) and save configuration metadata.  
This metadata is used for analysis and reproducibility.

In [6]:

import json

from helpers.decile_utils import (
    compute_corpus_boundaries,
    boundaries_to_metadata,
    print_boundaries,
)

metadata_path = RESULTS_DIR / "metadata.json"

if metadata_path.exists():
    print(f"⏭  Skipping — metadata already exists at {metadata_path}")
    with open(metadata_path) as f:
        metadata = json.load(f)
    print(f"   Collection: {metadata.get('collection_name', '?')}")
    print(f"   Corpus found: {metadata.get('corpus_found', '?')}")
else:
    print(metadata_path)
    print("Calculating decile boundaries from corpus...\n")

    # Locate corpus file
    CORPUS_PATH = COLLECTION_ROOT / "wiki_corpus.parquet"

    if not CORPUS_PATH.exists():
        print(f"⚠️  Corpus not found at {CORPUS_PATH}")
        print("   Metadata will be saved without decile boundaries")
        metadata = {
            "collection_name": COLLECTION_NAME,
            "output_name": OUTPUT_NAME,
            "embedding_model": EMBEDDING_MODEL,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "strategies": ALL_STRATEGIES,
            "top_k": TOP_K,
            "k_values_detailed": K_VALUES_DETAILED,
            "num_candidates": NUM_CANDIDATES,
            "num_questions": len(qa_df),
            "corpus_path": str(CORPUS_PATH),
            "corpus_found": False,
        }
    else:
        print(f"Reading corpus: {CORPUS_PATH}")
        
        # Use standardised boundary computation from rag.decile_utils
        boundaries_uw, boundaries_cw, stats, _ = compute_corpus_boundaries(
            corpus_path=CORPUS_PATH,
            batch_size=100_000,
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
        )
        
        print(f"\n✓ Calculated decile boundaries")
        print(f"  Unique documents: {stats['unique_documents_with_popularity']:,}")
        print(f"  Total chunks: {stats['total_chunks_after_splitting']:,}")
        print_boundaries(boundaries_uw, boundaries_cw)
        
        # Build metadata using standardised helper
        metadata = {
            "collection_name": COLLECTION_NAME,
            "output_name": OUTPUT_NAME,
            "embedding_model": EMBEDDING_MODEL,
            "chunk_size": CHUNK_SIZE,
            "chunk_overlap": CHUNK_OVERLAP,
            "strategies": ALL_STRATEGIES,
            "top_k": TOP_K,
            "k_values_detailed": K_VALUES_DETAILED,
            "num_candidates": NUM_CANDIDATES,
            "num_questions": len(qa_df),
            "corpus_path": str(CORPUS_PATH),
            "corpus_found": True,
            **boundaries_to_metadata(boundaries_uw, boundaries_cw, stats, CHUNK_SIZE, CHUNK_OVERLAP),
        }

    with open(metadata_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\n✅ Metadata saved to: {metadata_path}")
    print(f"\nReady for evaluation! Open rag_evaluation.ipynb to analyze results.")



/Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_full_bil_100/metadata.json
Calculating decile boundaries from corpus...

Reading corpus: /Users/cyro/Documents/VSC/PopularityBias/data/wiki_full_bil/wiki_corpus.parquet


Corpus boundaries:   0%|          | 0/60 [00:00<?, ?it/s]


✓ Calculated decile boundaries
  Unique documents: 5,890,044
  Total chunks: 24,651,978

  Unweighted (equal doc):
    Decile 0 (shown as 1): [1.0000, 8.1875)
    Decile 1 (shown as 2): [8.1875, 15.1250)
    Decile 2 (shown as 3): [15.1250, 25.3750)
    Decile 3 (shown as 4): [25.3750, 41.7708)
    Decile 4 (shown as 5): [41.7708, 69.6042)
    Decile 5 (shown as 6): [69.6042, 120.2708)
    Decile 6 (shown as 7): [120.2708, 224.0833)
    Decile 7 (shown as 8): [224.0833, 480.1458)
    Decile 8 (shown as 9): [480.1458, 1437.6666)
    Decile 9 (shown as 10): [1437.6666, 175763168.0000)

  Chunk-weighted (equal chunk):
    Decile 0 (shown as 1): [1.0000, 23.6667)
    Decile 1 (shown as 2): [23.6667, 55.7292)
    Decile 2 (shown as 3): [55.7292, 112.7083)
    Decile 3 (shown as 4): [112.7083, 217.6667)
    Decile 4 (shown as 5): [217.6667, 415.1250)
    Decile 5 (shown as 6): [415.1250, 810.4583)
    Decile 6 (shown as 7): [810.4583, 1674.5416)
    Decile 7 (shown as 8): [1674.5416, 3895.0